In [ ]:
%cd ../..
import os
import torch
from tqdm import tqdm
import pydicom
from glob import glob
from omegaconf import OmegaConf

from dinov2.inference import generate_embeddings, build_model, view_volume, is_HU

In [ ]:
def load_dicom_undersample(folder_path: str, max_samples=200):

    dicom_files = [
        os.path.join(folder_path, x)
        for x in os.listdir(folder_path)
    ]
    dicom_files.sort(key=lambda x: int(x.split("/")[-1].replace(".dcm", "").replace("I", "")))
    if len(dicom_files) > max_samples:
        r = len(dicom_files) // max_samples
        dicom_files = dicom_files[::r]
    else:
        r = 1
        
    slices = []
    for filepath in dicom_files:
        dataset = pydicom.dcmread(filepath)
        slices.append(dataset)

    slices.sort(key=lambda s: float(s.ImagePositionPatient[2]))

    z_spacing = float(
        slices[1].ImagePositionPatient[2] - slices[0].ImagePositionPatient[2]
    )
    x_spacing = float(slices[0].PixelSpacing[0])
    y_spacing = float(slices[0].PixelSpacing[1])

    spacing = (z_spacing, x_spacing, y_spacing)

    rescale_slope = slices[0].RescaleSlope
    rescale_intercept = slices[0].RescaleIntercept

    data_stack = []
    for i, s in enumerate(slices):
        data_stack.append(torch.from_numpy(s.pixel_array).float())

    image = torch.stack(data_stack)
    image = rescale_slope * image + rescale_intercept
    image = torch.clip(image, -1000, 1900)

    assert is_HU(image)
    
    return image, spacing

In [ ]:
sample_path = "/scratch/VM/radio-foundation/datasets/COVID-CT-DS/Original-DICOM/Normal/P268/SER00005"
img, spacing = load_dicom_undersample(sample_path)
view_volume(img, spacing)

print(img.shape, img.min(), img.max())

In [ ]:
config_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/config.yaml"
checkpoint_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/eval/training_99999/teacher_checkpoint.pth"

device = torch.device("cuda")

config = OmegaConf.load(config_path)
model, autocast_ctx = build_model(checkpoint_path, config, img_size=504, device=device)

In [ ]:
data_path = "/scratch/VM/radio-foundation/datasets/COVID-CT-DS/Original-DICOM"
output_path = "/scratch/VM/radio-foundation/cache/embeddings/COVID-CT-DS"

In [ ]:
data_kwargs = dict(
    fmean = -573.8,
    fstd = 461.3,
    channels = 10,
    img_size = 504,
    patch_size = 14,
    device="cuda",
    block_size=64,
    autocast_ctx=autocast_ctx,
)
class_names = ["Fabrosis", "Mild", "Moderate", "Normal", "Severe"]
for c in class_names:
    print(c)
    base_path = os.path.join(data_path, c)
    os.makedirs(os.path.join(output_path, c), exist_ok=True)

    series_paths = glob(os.path.join(base_path, "**/"), recursive=True)
    series_paths = [d for d in series_paths if glob(os.path.join(d, "*.dcm")) or glob(os.path.join(d, "I*"))]

    for path in tqdm(series_paths):
        series_id = path.split("/")[-3]
            
        p_output_dir = os.path.join(output_path, c, f"{series_id}.pth")
        if os.path.exists(p_output_dir):
            continue

        try:
            img, spacing = load_dicom_undersample(folder_path=path)
            img = img.clip(-1000, 200)

            collated_features = generate_embeddings(
            img,
            model=model,
            **data_kwargs # type: ignore
            )

            output = {"cls": collated_features["cls"]}

            torch.save(output, p_output_dir)
        except Exception:
            pass
